In [ ]:
!pip install pandas scikit-learn plotly

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import  OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
# setting Jedha color palette as default
pio.templates["jedha"] = go.layout.Template(
    layout_colorway=["#4B9AC7", "#4BE8E0", "#9DD4F3", "#97FBF6", "#2A7FAF", "#23B1AB", "#0E3449", "#015955"]
)
pio.templates.default = "jedha"
pio.renderers.default = "svg" # to be replaced by "iframe" if working on JULIE


In [ ]:
# Import enedis1
print("Loading enedis1...")
enedis1 = pd.read_csv("../data/extract_cvs_engis_dataset.csv", sep=";",
    lineterminator="\n",
    encoding="utf-8")
print("...Done.")
print()

Loading enedis1...
...Done.



In [ ]:
# Basic stats
data_desc = enedis1.describe(include='all')
print(enedis1.shape)
data_desc


(50000, 22)


,secteur_activite,jour_max_du_mois_0_1,horodate,semaine_max_du_mois_0_1,plage_de_puissance_souscrite,region,total_energie_soutiree_wh,nb_points_soutirage,date,ville,...,zone_a,zone_b,zone_c,vacances_zone_a,vacances_zone_b,vacances_zone_c,nom_vacances,temperature_2m_mean,relative_humidity_mean,precipitation_sum
count,50000,50000.0,50000,50000.0,50000,50000,4.442200e+04,50000.000000,50000,41600,...,41600,41600,41600,50000,50000,50000,10000,41600.000000,41600.000000,41600.000000
unique,4,NaN,82,NaN,7,12,NaN,NaN,20,10,...,2,2,2,2,2,2,1,NaN,NaN,NaN
top,S1: Agriculture,NaN,2025-06-30T22:00:00Z,NaN,P1: ]36-120] kVA,Auvergne-Rhône-Alpes,NaN,NaN,2025-01-01,Lyon,...,True,False,False,False,False,False,Vacances de Noël,NaN,NaN,NaN
freq,12917,NaN,672,NaN,7160,4452,NaN,NaN,3656,4452,...,21212,25160,37652,40000,40000,40000,10000,NaN,NaN,NaN
mean,NaN,0.0,NaN,0.0,NaN,NaN,2.191475e+07,1570.215320,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14.517543,74.543173,2.147144
std,NaN,0.0,NaN,0.0,NaN,NaN,5.094391e+07,5967.956761,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.582715,10.967533,4.431991
min,NaN,0.0,NaN,0.0,NaN,NaN,0.000000e+00,0.000000,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.306250,37.619495,0.000000
25%,NaN,0.0,NaN,0.0,NaN,NaN,0.000000e+00,1.000000,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.293750,69.994050,0.000000
50%,NaN,0.0,NaN,0.0,NaN,NaN,1.851860e+06,58.000000,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.835416,76.711960,0.100000
75%,NaN,0.0,NaN,0.0,NaN,NaN,1.942364e+07,867.000000,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17.852083,81.716600,2.600000


In [ ]:
# Import enedis2
print("Loading enedis2...")
enedis2 = pd.read_csv("../data/extract_cvs_engis_dataset_500000.csv", sep=";",
    lineterminator="\n",
    encoding="utf-8")
print("...Done.")
print()

Loading enedis2...
...Done.



In [ ]:
# Basic stats
data_desc = enedis2.describe(include='all')
print(enedis2.shape)
data_desc


(550000, 22)


,secteur_activite,jour_max_du_mois_0_1,horodate,semaine_max_du_mois_0_1,plage_de_puissance_souscrite,region,total_energie_soutiree_wh,nb_points_soutirage,date,ville,...,zone_a,zone_b,zone_c,vacances_zone_a,vacances_zone_b,vacances_zone_c,nom_vacances,temperature_2m_mean,relative_humidity_mean,precipitation_sum
count,550000,550000.0,550000,550000.0,550000,550000,4.882730e+05,550000.000000,550000,458188,...,458188,458188,458188,550000,550000,550000,110000,458188.000000,458188.000000,458188.000000
unique,4,NaN,848,NaN,7,12,NaN,NaN,31,10,...,2,2,2,2,2,2,1,NaN,NaN,NaN
top,S1: Agriculture,NaN,2024-07-02T14:00:00Z,NaN,P1: ]36-120] kVA,Auvergne-Rhône-Alpes,NaN,NaN,2024-07-01,Lyon,...,False,False,False,False,False,False,Vacances de Noël,NaN,NaN,NaN
freq,142084,NaN,672,NaN,78598,45948,NaN,NaN,32256,45948,...,229098,274996,412282,440000,440000,440000,110000,NaN,NaN,NaN
mean,NaN,0.0,NaN,0.0,NaN,NaN,2.763204e+07,1578.696511,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14.556534,73.522792,2.504267
std,NaN,0.0,NaN,0.0,NaN,NaN,6.446021e+07,6032.111711,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.376747,11.872584,5.078139
min,NaN,0.0,NaN,0.0,NaN,NaN,0.000000e+00,0.000000,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.306250,35.469093,0.000000
25%,NaN,0.0,NaN,0.0,NaN,NaN,0.000000e+00,1.000000,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.793750,67.996200,0.000000
50%,NaN,0.0,NaN,0.0,NaN,NaN,1.992465e+06,58.000000,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.372916,76.052600,0.100000
75%,NaN,0.0,NaN,0.0,NaN,NaN,2.455241e+07,867.000000,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18.281250,81.716600,2.600000


In [ ]:
enedis1.head()

,secteur_activite,jour_max_du_mois_0_1,horodate,semaine_max_du_mois_0_1,plage_de_puissance_souscrite,region,total_energie_soutiree_wh,nb_points_soutirage,date,ville,...,zone_a,zone_b,zone_c,vacances_zone_a,vacances_zone_b,vacances_zone_c,nom_vacances,temperature_2m_mean,relative_humidity_mean,precipitation_sum
0,S1: Agriculture,0,2025-06-30T22:00:00Z,0,P1: ]36-120] kVA,Auvergne-Rhône-Alpes,15727584.0,1746,2025-06-30,Lyon,...,True,False,False,False,False,False,NaN,29.752083,46.599354,0.0
1,S1: Agriculture,0,2024-12-31T23:00:00Z,0,P1: ]36-120] kVA,Auvergne-Rhône-Alpes,4151369.0,1733,2024-12-31,Lyon,...,True,False,False,True,True,True,Vacances de Noël,-1.279167,91.467995,0.0
2,S1: Agriculture,0,2025-09-30T22:00:00Z,0,P1: ]36-120] kVA,Auvergne-Rhône-Alpes,3873704.0,1750,2025-09-30,Lyon,...,True,False,False,False,False,False,NaN,12.895833,81.129370,0.0
3,S1: Agriculture,0,2024-06-30T22:00:00Z,0,P1: ]36-120] kVA,Auvergne-Rhône-Alpes,4680196.0,1739,2024-06-30,Lyon,...,True,False,False,False,False,False,NaN,19.995832,75.284070,0.7
4,S1: Agriculture,0,2024-09-30T22:00:00Z,0,P1: ]36-120] kVA,Auvergne-Rhône-Alpes,3803091.0,1738,2024-09-30,Lyon,...,True,False,False,False,False,False,NaN,16.581251,63.248806,0.3


In [ ]:
enedis2.head()

,secteur_activite,jour_max_du_mois_0_1,horodate,semaine_max_du_mois_0_1,plage_de_puissance_souscrite,region,total_energie_soutiree_wh,nb_points_soutirage,date,ville,...,zone_a,zone_b,zone_c,vacances_zone_a,vacances_zone_b,vacances_zone_c,nom_vacances,temperature_2m_mean,relative_humidity_mean,precipitation_sum
0,S1: Agriculture,0,2025-06-30T22:00:00Z,0,P1: ]36-120] kVA,Auvergne-Rhône-Alpes,15727584.0,1746,2025-06-30,Lyon,...,True,False,False,False,False,False,NaN,29.752083,46.599354,0.0
1,S1: Agriculture,0,2024-12-31T23:00:00Z,0,P1: ]36-120] kVA,Auvergne-Rhône-Alpes,4151369.0,1733,2024-12-31,Lyon,...,True,False,False,True,True,True,Vacances de Noël,-1.279167,91.467995,0.0
2,S1: Agriculture,0,2025-09-30T22:00:00Z,0,P1: ]36-120] kVA,Auvergne-Rhône-Alpes,3873704.0,1750,2025-09-30,Lyon,...,True,False,False,False,False,False,NaN,12.895833,81.129370,0.0
3,S1: Agriculture,0,2024-06-30T22:00:00Z,0,P1: ]36-120] kVA,Auvergne-Rhône-Alpes,4680196.0,1739,2024-06-30,Lyon,...,True,False,False,False,False,False,NaN,19.995832,75.284070,0.7
4,S1: Agriculture,0,2024-09-30T22:00:00Z,0,P1: ]36-120] kVA,Auvergne-Rhône-Alpes,3803091.0,1738,2024-09-30,Lyon,...,True,False,False,False,False,False,NaN,16.581251,63.248806,0.3


In [ ]:
enedis2.tail()

,secteur_activite,jour_max_du_mois_0_1,horodate,semaine_max_du_mois_0_1,plage_de_puissance_souscrite,region,total_energie_soutiree_wh,nb_points_soutirage,date,ville,...,zone_a,zone_b,zone_c,vacances_zone_a,vacances_zone_b,vacances_zone_c,nom_vacances,temperature_2m_mean,relative_humidity_mean,precipitation_sum
549995,S4: Non Affecté,0,2025-01-02T15:30:00Z,0,P2: ]120-250] kVA,Pays de la Loire,71750.0,4,2025-01-02,Nantes,...,False,True,False,True,True,True,Vacances de Noël,7.972917,92.029330,14.600000
549996,S4: Non Affecté,0,2025-10-02T14:30:00Z,0,P2: ]120-250] kVA,Pays de la Loire,NaN,1,2025-10-02,Nantes,...,False,True,False,False,False,False,NaN,15.868751,63.061780,0.000000
549997,S4: Non Affecté,0,2025-07-02T14:30:00Z,0,P2: ]120-250] kVA,Pays de la Loire,NaN,1,2025-07-02,Nantes,...,False,True,False,False,False,False,NaN,22.477083,60.468067,0.000000
549998,S4: Non Affecté,0,2023-10-02T14:30:00Z,0,P2: ]120-250] kVA,Pays de la Loire,80500.0,6,2023-10-02,Nantes,...,False,True,False,False,False,False,NaN,19.816668,82.107420,0.000000
549999,S4: Non Affecté,0,2024-04-02T14:30:00Z,0,P2: ]120-250] kVA,Pays de la Loire,116166.0,7,2024-04-02,Nantes,...,False,True,False,False,False,False,NaN,10.597917,83.999535,11.400001


In [ ]:
# Drop useless columns / columns with too many missing values
useless_cols = ["jour_max_du_mois_0_1", "semaine_max_du_mois_0_1", "horodate", "zone_a", "zone_b", "zone_c", "nom_vacances"]

print("Dropping useless columns...")
enedis = enedis2.drop(
    useless_cols, axis=1
)  # axis = 1 indicates that we are dropping along the column axis
# never hesitate to look at a function's documentation using the command name_of_the_function?
print("...Done.")
print(enedis.head())

Dropping useless columns...
...Done.
  secteur_activite plage_de_puissance_souscrite                region  \
0  S1: Agriculture             P1: ]36-120] kVA  Auvergne-Rhône-Alpes   
1  S1: Agriculture             P1: ]36-120] kVA  Auvergne-Rhône-Alpes   
2  S1: Agriculture             P1: ]36-120] kVA  Auvergne-Rhône-Alpes   
3  S1: Agriculture             P1: ]36-120] kVA  Auvergne-Rhône-Alpes   
4  S1: Agriculture             P1: ]36-120] kVA  Auvergne-Rhône-Alpes   

   total_energie_soutiree_wh  nb_points_soutirage        date ville     lat  \
0                 15727584.0                 1746  2025-06-30  Lyon  45.764   
1                  4151369.0                 1733  2024-12-31  Lyon  45.764   
2                  3873704.0                 1750  2025-09-30  Lyon  45.764   
3                  4680196.0                 1739  2024-06-30  Lyon  45.764   
4                  3803091.0                 1738  2024-09-30  Lyon  45.764   

      lon  vacances_zone_a  vacances_zone_b  vaca

In [ ]:
# Drop useless columns / columns with too many missing values
useless_cols = ["lat", "lon"]

print("Dropping useless columns...")
enedis = enedis.drop(
    useless_cols, axis=1
)  # axis = 1 indicates that we are dropping along the column axis
# never hesitate to look at a function's documentation using the command name_of_the_function?
print("...Done.")
print(enedis.head())

Dropping useless columns...
...Done.
  secteur_activite plage_de_puissance_souscrite                region  \
0  S1: Agriculture             P1: ]36-120] kVA  Auvergne-Rhône-Alpes   
1  S1: Agriculture             P1: ]36-120] kVA  Auvergne-Rhône-Alpes   
2  S1: Agriculture             P1: ]36-120] kVA  Auvergne-Rhône-Alpes   
3  S1: Agriculture             P1: ]36-120] kVA  Auvergne-Rhône-Alpes   
4  S1: Agriculture             P1: ]36-120] kVA  Auvergne-Rhône-Alpes   

   total_energie_soutiree_wh  nb_points_soutirage        date ville  \
0                 15727584.0                 1746  2025-06-30  Lyon   
1                  4151369.0                 1733  2024-12-31  Lyon   
2                  3873704.0                 1750  2025-09-30  Lyon   
3                  4680196.0                 1739  2024-06-30  Lyon   
4                  3803091.0                 1738  2024-09-30  Lyon   

   vacances_zone_a  vacances_zone_b  vacances_zone_c  temperature_2m_mean  \
0            False  

In [ ]:
enedis.dtypes

,0
secteur_activite,object
plage_de_puissance_souscrite,object
region,object
total_energie_soutiree_wh,float64
nb_points_soutirage,int64
date,object
ville,object
vacances_zone_a,bool
vacances_zone_b,bool
vacances_zone_c,bool


In [ ]:
print(enedis.isnull().sum())

secteur_activite                    0
plage_de_puissance_souscrite        0
region                              0
total_energie_soutiree_wh       61727
nb_points_soutirage                 0
date                                0
ville                           91812
vacances_zone_a                     0
vacances_zone_b                     0
vacances_zone_c                     0
temperature_2m_mean             91812
relative_humidity_mean          91812
precipitation_sum               91812
dtype: int64


In [ ]:
# Drop lines containing no value (using masks)

print("Dropping NaN in temperature_2m_mean...")
to_keep = enedis["temperature_2m_mean"].notnull()
enedis = enedis.loc[to_keep, :]
print("Done. Number of lines remaining : ", enedis.shape[0])
print()

enedis.head()

Dropping NaN in temperature_2m_mean...
Done. Number of lines remaining :  458188



,secteur_activite,plage_de_puissance_souscrite,region,total_energie_soutiree_wh,nb_points_soutirage,date,ville,vacances_zone_a,vacances_zone_b,vacances_zone_c,temperature_2m_mean,relative_humidity_mean,precipitation_sum
0,S1: Agriculture,P1: ]36-120] kVA,Auvergne-Rhône-Alpes,15727584.0,1746,2025-06-30,Lyon,False,False,False,29.752083,46.599354,0.0
1,S1: Agriculture,P1: ]36-120] kVA,Auvergne-Rhône-Alpes,4151369.0,1733,2024-12-31,Lyon,True,True,True,-1.279167,91.467995,0.0
2,S1: Agriculture,P1: ]36-120] kVA,Auvergne-Rhône-Alpes,3873704.0,1750,2025-09-30,Lyon,False,False,False,12.895833,81.129370,0.0
3,S1: Agriculture,P1: ]36-120] kVA,Auvergne-Rhône-Alpes,4680196.0,1739,2024-06-30,Lyon,False,False,False,19.995832,75.284070,0.7
4,S1: Agriculture,P1: ]36-120] kVA,Auvergne-Rhône-Alpes,3803091.0,1738,2024-09-30,Lyon,False,False,False,16.581251,63.248806,0.3


In [ ]:
# Basic stats
data_desc = enedis.describe(include='all')
print(enedis.shape)
data_desc

(458188, 13)


,secteur_activite,plage_de_puissance_souscrite,region,total_energie_soutiree_wh,nb_points_soutirage,date,ville,vacances_zone_a,vacances_zone_b,vacances_zone_c,temperature_2m_mean,relative_humidity_mean,precipitation_sum
count,458188,458188,458188,4.095520e+05,458188.000000,458188,458188,458188,458188,458188,458188.000000,458188.000000,458188.000000
unique,4,7,10,NaN,NaN,31,10,2,2,2,NaN,NaN,NaN
top,S1: Agriculture,P1: ]36-120] kVA,Auvergne-Rhône-Alpes,NaN,NaN,2023-07-01,Lyon,False,False,False,NaN,NaN,NaN
freq,118368,65482,45948,NaN,NaN,26880,45948,366556,366556,366556,NaN,NaN,NaN
mean,NaN,NaN,NaN,2.779391e+07,1605.165511,NaN,NaN,NaN,NaN,NaN,14.556534,73.522792,2.504267
std,NaN,NaN,NaN,6.655748e+07,6319.004630,NaN,NaN,NaN,NaN,NaN,6.376747,11.872584,5.078139
min,NaN,NaN,NaN,0.000000e+00,0.000000,NaN,NaN,NaN,NaN,NaN,-1.306250,35.469093,0.000000
25%,NaN,NaN,NaN,0.000000e+00,1.000000,NaN,NaN,NaN,NaN,NaN,9.793750,67.996200,0.000000
50%,NaN,NaN,NaN,1.799906e+06,60.000000,NaN,NaN,NaN,NaN,NaN,13.372916,76.052600,0.100000
75%,NaN,NaN,NaN,2.373098e+07,831.000000,NaN,NaN,NaN,NaN,NaN,18.281250,81.716600,2.600000


In [ ]:
# Drop lines containing no value (using masks)

print("Dropping NaN in total_energie_soutiree_wh...")
to_keep = enedis["total_energie_soutiree_wh"].notnull()
enedis = enedis.loc[to_keep, :]
print("Done. Number of lines remaining : ", enedis.shape[0])
print()

enedis.head()

Dropping NaN in total_energie_soutiree_wh...
Done. Number of lines remaining :  409552



,secteur_activite,plage_de_puissance_souscrite,region,total_energie_soutiree_wh,nb_points_soutirage,date,ville,vacances_zone_a,vacances_zone_b,vacances_zone_c,temperature_2m_mean,relative_humidity_mean,precipitation_sum
0,S1: Agriculture,P1: ]36-120] kVA,Auvergne-Rhône-Alpes,15727584.0,1746,2025-06-30,Lyon,False,False,False,29.752083,46.599354,0.0
1,S1: Agriculture,P1: ]36-120] kVA,Auvergne-Rhône-Alpes,4151369.0,1733,2024-12-31,Lyon,True,True,True,-1.279167,91.467995,0.0
2,S1: Agriculture,P1: ]36-120] kVA,Auvergne-Rhône-Alpes,3873704.0,1750,2025-09-30,Lyon,False,False,False,12.895833,81.129370,0.0
3,S1: Agriculture,P1: ]36-120] kVA,Auvergne-Rhône-Alpes,4680196.0,1739,2024-06-30,Lyon,False,False,False,19.995832,75.284070,0.7
4,S1: Agriculture,P1: ]36-120] kVA,Auvergne-Rhône-Alpes,3803091.0,1738,2024-09-30,Lyon,False,False,False,16.581251,63.248806,0.3


In [ ]:
# Basic stats
data_desc = enedis.describe(include='all')
print(enedis.shape)
data_desc

(409552, 13)


,secteur_activite,plage_de_puissance_souscrite,region,total_energie_soutiree_wh,nb_points_soutirage,date,ville,vacances_zone_a,vacances_zone_b,vacances_zone_c,temperature_2m_mean,relative_humidity_mean,precipitation_sum
count,409552,409552,409552,4.095520e+05,409552.000000,409552,409552,409552,409552,409552,409552.000000,409552.000000,409552.000000
unique,4,7,10,NaN,NaN,31,10,2,2,2,NaN,NaN,NaN
top,S3: Tertiaire,P3: Total ]36-250] kVA,Occitanie,NaN,NaN,2025-04-01,Toulouse,False,False,False,NaN,NaN,NaN
freq,115553,62432,43224,NaN,NaN,24294,43224,328114,328114,328114,NaN,NaN,NaN
mean,NaN,NaN,NaN,2.779391e+07,1794.339454,NaN,NaN,NaN,NaN,NaN,14.540970,73.545931,2.496518
std,NaN,NaN,NaN,6.655748e+07,6658.373470,NaN,NaN,NaN,NaN,NaN,6.366185,11.878868,5.071131
min,NaN,NaN,NaN,0.000000e+00,0.000000,NaN,NaN,NaN,NaN,NaN,-1.306250,35.469093,0.000000
25%,NaN,NaN,NaN,0.000000e+00,0.000000,NaN,NaN,NaN,NaN,NaN,9.793750,67.996200,0.000000
50%,NaN,NaN,NaN,1.799906e+06,93.000000,NaN,NaN,NaN,NaN,NaN,13.345833,76.052600,0.100000
75%,NaN,NaN,NaN,2.373098e+07,931.000000,NaN,NaN,NaN,NaN,NaN,18.475000,81.716600,2.600000


In [ ]:
# Drop useless columns / columns with too many missing values
useless_cols = ["region"]

print("Dropping useless columns...")
enedis = enedis.drop(
    useless_cols, axis=1
)  # axis = 1 indicates that we are dropping along the column axis
# never hesitate to look at a function's documentation using the command name_of_the_function?
print("...Done.")
print(enedis.head())

Dropping useless columns...
...Done.
  secteur_activite plage_de_puissance_souscrite  total_energie_soutiree_wh  \
0  S1: Agriculture             P1: ]36-120] kVA                 15727584.0   
1  S1: Agriculture             P1: ]36-120] kVA                  4151369.0   
2  S1: Agriculture             P1: ]36-120] kVA                  3873704.0   
3  S1: Agriculture             P1: ]36-120] kVA                  4680196.0   
4  S1: Agriculture             P1: ]36-120] kVA                  3803091.0   

   nb_points_soutirage        date ville  vacances_zone_a  vacances_zone_b  \
0                 1746  2025-06-30  Lyon            False            False   
1                 1733  2024-12-31  Lyon             True             True   
2                 1750  2025-09-30  Lyon            False            False   
3                 1739  2024-06-30  Lyon            False            False   
4                 1738  2024-09-30  Lyon            False            False   

   vacances_zone_c  tempe

In [ ]:
# Separate target variable Y from features X
print("Separating labels from features...")
features_list = ["secteur_activite", "plage_de_puissance_souscrite", "nb_points_soutirage", "date", "ville", "vacances_zone_a",	"vacances_zone_b", "vacances_zone_c",	"temperature_2m_mean", "relative_humidity_mean",	"precipitation_sum"]
target_variable = "total_energie_soutiree_wh"

X = enedis.loc[:,features_list]
Y = enedis.loc[:,target_variable]

print("...Done.")
print()

print('Y : ')
print(Y.head())
print()
print('X :')
print(X.head())


Separating labels from features...
...Done.

Y : 
0    15727584.0
1     4151369.0
2     3873704.0
3     4680196.0
4     3803091.0
Name: total_energie_soutiree_wh, dtype: float64

X :
  secteur_activite plage_de_puissance_souscrite  nb_points_soutirage  \
0  S1: Agriculture             P1: ]36-120] kVA                 1746   
1  S1: Agriculture             P1: ]36-120] kVA                 1733   
2  S1: Agriculture             P1: ]36-120] kVA                 1750   
3  S1: Agriculture             P1: ]36-120] kVA                 1739   
4  S1: Agriculture             P1: ]36-120] kVA                 1738   

         date ville  vacances_zone_a  vacances_zone_b  vacances_zone_c  \
0  2025-06-30  Lyon            False            False            False   
1  2024-12-31  Lyon             True             True             True   
2  2025-09-30  Lyon            False            False            False   
3  2024-06-30  Lyon            False            False            False   
4  2024-09-30 

In [ ]:
# Automatically detect names of numeric/categorical columns
numeric_features = []
categorical_features = []
for i,t in X.dtypes.items():
    if ('float' in str(t)) or ('int' in str(t)) :
        numeric_features.append(i)
    else :
        categorical_features.append(i)

print('Found numeric features ', numeric_features)
print('Found categorical features ', categorical_features)

Found numeric features  ['nb_points_soutirage', 'temperature_2m_mean', 'relative_humidity_mean', 'precipitation_sum']
Found categorical features  ['secteur_activite', 'plage_de_puissance_souscrite', 'date', 'ville', 'vacances_zone_a', 'vacances_zone_b', 'vacances_zone_c']


In [ ]:
# Divide dataset Train set & Test set
print("Dividing into train and test sets...")
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=0)
print("...Done.")
print()


Dividing into train and test sets...
...Done.



In [ ]:
# Create pipeline for numeric features
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')), # missing values will be replaced by columns' mean
    ('scaler', StandardScaler())
])


In [ ]:
# Create pipeline for categorical features
categorical_transformer = Pipeline(
    steps=[
    ('encoder', OneHotEncoder(drop='first')) # first column will be dropped to avoid creating correlations between features
    ])


In [ ]:
# Use ColumnTransformer to make a preprocessor object that describes all the treatments to be done
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])


In [ ]:
# Preprocessings on train set
print("Performing preprocessings on train set...")
print(X_train.head())
X_train = preprocessor.fit_transform(X_train)
print('...Done.')
print(X_train[0:5]) # MUST use this syntax because X_train is a numpy array and not a pandas DataFrame anymore
print()

# Preprocessings on test set
print("Performing preprocessings on test set...")
print(X_test.head())
X_test = preprocessor.transform(X_test) # Don't fit again !! The test set is used for validating decisions
# we made based on the training set, therefore we can only apply transformations that were parametered using the training set.
# Otherwise this creates what is called a leak from the test set which will introduce a bias in all your results.
print('...Done.')
print(X_test[0:5,:]) # MUST use this syntax because X_test is a numpy array and not a pandas DataFrame anymore
print()


Performing preprocessings on train set...
       secteur_activite plage_de_puissance_souscrite  nb_points_soutirage  \
321064    S2: Industrie          P5: ]1000-2000] kVA                    0   
219420    S3: Tertiaire           P4: ]250-1000] kVA                 1027   
497504    S2: Industrie               P6: > 2000 kVA                    0   
484506    S2: Industrie       P3: Total ]36-250] kVA                 3331   
466920  S4: Non Affecté           P4: ]250-1000] kVA                    0   

              date     ville  vacances_zone_a  vacances_zone_b  \
321064  2024-04-01  Toulouse            False            False   
219420  2025-01-01     Rouen             True             True   
497504  2025-01-02      Lyon             True             True   
484506  2025-07-02     Dijon            False            False   
466920  2025-07-02     Lille            False            False   

        vacances_zone_c  temperature_2m_mean  relative_humidity_mean  \
321064            False   

In [ ]:
# Train model
print("Train model...")
regressor = LinearRegression()
regressor.fit(X_train, Y_train)
print("...Done.")


Train model...
...Done.


In [ ]:
# Predictions on training set
print("Predictions on training set...")
Y_train_pred = regressor.predict(X_train)
print("...Done.")
print(Y_train_pred)
print()


Predictions on training set...
...Done.
[17475795.55905472 36223980.50961146 49504523.66752763 ...
 12364643.34558514 51678615.29396772 62761063.97608633]



In [ ]:
# Predictions on test set
print("Predictions on test set...")
Y_test_pred = regressor.predict(X_test)
print("...Done.")
print(Y_test_pred)
print()


Predictions on test set...
...Done.
[-14334200.78956084  -6975109.81592272  -8874780.73854778 ...
  45629720.26366743  29704792.43653539  80733151.89518146]



In [ ]:
# Print R^2 scores
print("R2 score on training set : ", r2_score(Y_train, Y_train_pred))
print("R2 score on test set : ", r2_score(Y_test, Y_test_pred))


R2 score on training set :  0.39607366393949095
R2 score on test set :  0.40512193160939314


In [ ]:
regressor.coef_


array([29523744.67459669,   320722.40272209,  -581789.4433804 ,
         -97878.7350478 , 30842776.09067148, 35232711.8862914 ,
       -2296074.23638275, 11803006.37371428, 11629261.16203707,
       25642445.52036235, 15293951.82736543, 23818524.01833255,
       62548959.85158521,   241533.95683652, -2218310.34390412,
       -3276280.87643998, -2231982.38491468,  7920720.45674389,
       -3345872.2719419 , -2800047.4657275 ,  4312840.08660031,
       -1383759.54135832,   301403.88080206, 10491487.86879218,
       -3056223.41595722,  6606642.53340271,  9471063.7355537 ,
       -1762295.55640128,  8626246.52256272,  8262957.74368453,
         586498.26119366,  -971146.17396141, -1736671.08713396,
        5485023.62502312,   571388.89063491,  7297024.20946499,
        9379115.28376511,  2398659.46489034, 10594968.50476112,
       11835569.6700304 ,   422733.5909077 ,  7051070.40606281,
        9080731.02420783, 13171485.74189771, 20389671.58502664,
        3954448.73033719,  6205366.55765

In [ ]:
column_names = []
for name, pipeline, features_list in preprocessor.transformers_: # loop over pipelines
    if name == 'num': # if pipeline is for numeric variables
        features = features_list # just get the names of columns to which it has been applied
    else: # if pipeline is for categorical variables
        features = pipeline.named_steps['encoder'].get_feature_names_out() # get output columns names from OneHotEncoder
    column_names.extend(features) # concatenate features names

print("Names of columns corresponding to each coefficient: ", column_names)


Names of columns corresponding to each coefficient:  ['nb_points_soutirage', 'temperature_2m_mean', 'relative_humidity_mean', 'precipitation_sum', 'secteur_activite_S2: Industrie', 'secteur_activite_S3: Tertiaire', 'secteur_activite_S4: Non Affecté', 'plage_de_puissance_souscrite_P2: ]120-250] kVA', 'plage_de_puissance_souscrite_P3: Total ]36-250] kVA', 'plage_de_puissance_souscrite_P4: ]250-1000] kVA', 'plage_de_puissance_souscrite_P5: ]1000-2000] kVA', 'plage_de_puissance_souscrite_P6: > 2000 kVA', 'plage_de_puissance_souscrite_P7: Total > 250 kVA', 'date_2023-07-01', 'date_2023-07-02', 'date_2023-09-30', 'date_2023-10-01', 'date_2023-10-02', 'date_2023-12-31', 'date_2024-01-01', 'date_2024-01-02', 'date_2024-03-31', 'date_2024-04-01', 'date_2024-04-02', 'date_2024-06-30', 'date_2024-07-01', 'date_2024-07-02', 'date_2024-09-30', 'date_2024-10-01', 'date_2024-10-02', 'date_2024-10-03', 'date_2024-12-31', 'date_2025-01-01', 'date_2025-01-02', 'date_2025-03-31', 'date_2025-04-01', 'date

In [ ]:
# Create a pandas DataFrame
coefs = pd.DataFrame(index = column_names, data = regressor.coef_.transpose(), columns=["coefficients"])
coefs


,coefficients
nb_points_soutirage,2.952374e+07
temperature_2m_mean,3.207224e+05
relative_humidity_mean,-5.817894e+05
precipitation_sum,-9.787874e+04
secteur_activite_S2: Industrie,3.084278e+07
secteur_activite_S3: Tertiaire,3.523271e+07
secteur_activite_S4: Non Affecté,-2.296074e+06
plage_de_puissance_souscrite_P2: ]120-250] kVA,1.180301e+07
plage_de_puissance_souscrite_P3: Total ]36-250] kVA,1.162926e+07
plage_de_puissance_souscrite_P4: ]250-1000] kVA,2.564245e+07


In [ ]:
# Compute abs() and sort values
feature_importance = abs(coefs).sort_values(by = 'coefficients')
feature_importance


,coefficients
precipitation_sum,9.787874e+04
date_2023-07-01,2.415340e+05
date_2024-04-01,3.014039e+05
temperature_2m_mean,3.207224e+05
date_2025-09-30,4.227336e+05
date_2025-03-31,5.713889e+05
relative_humidity_mean,5.817894e+05
date_2024-10-03,5.864983e+05
vacances_zone_b_True,9.441267e+05
vacances_zone_c_True,9.441267e+05
